# NeurIPS 2026 Smart Buildings — Version 19: Iterative Learning Control

**Experiment:** day-to-day, phase-aligned iterative learning control (ILC) on top of the fixed, physics-based V7/V15 feedback controller. The historical V7 public score is **0.03947**; V19 is **unscored**. The baseline uses V15's outdoor-temperature denormalization correction.

## Repeating task and update

A 30-day episode contains repeated **288-step (5-minute) daily cycles**. Each *completed* day provides zone temperature errors and actual applied feedforward offsets for the **next** day of the same schedule class (weekday vs weekend). Learned feedforward profiles are indexed by **zone × clock slot**, with no building IDs, training data, or external models.

For each mode-specific actuator-fraction correction at clock slot k, where $u_d$ is the **actually applied**, post-saturation offset (zero when gated off):

$$
u_{d+1}(k) = \operatorname{clip}\left(0.90\,u_d(k)+0.035\,e_d(k+3),\;-u_{\max},\;u_{\max}\right)
$$

The 3-step response offset (15 minutes) associates a temperature response with an earlier action. Cooling error is $T-(T_{\mathrm{high}}-0.2\,\mathrm{K})$; heating error is $(T_{\mathrm{low}}+0.2\,\mathrm{K})-T$. The first day uses **zero** learned correction. Updates happen **only at day boundaries**, not at every five-minute step. Each usable day must contain at least 240 aligned slots; the last three slots are not trained because their response falls into the next day.

## Safety and generalization limits

- The existing physical feedback controller always runs. ILC adds at most **±0.15 damper fraction** when a zone is actively cooling or **±0.18 valve fraction** when it is actively reheating. Learning does **not** turn on AHUs or hot water, change the AHU coil, or change the system topology.
- Applied corrections require a previously learned *same weekday/weekend* profile and outdoor air within **4 K** of the reference day at the same time. No profile means no change.
- No update occurs when time features are missing, clocks jump, sensor readings are invalid, or a daily trace is incomplete. Previously learned slots without fresh valid measurements are invalidated rather than carried forward. `reset()` clears **all** episode memory, preventing learning state from carrying between building/weather scenarios.
- The prior-day temperature error is not a perfect system gradient: lag, daily occupancy, envelope dynamics, and weather drift can make ILC ineffective or worse than V7/V15. Actuator-saturation and comfort-penalty effects require real simulator measurement.

## Components

| Component | Responsibility |
| --- | --- |
| `PhysicsBaselinePolicy` | Fixed V15/V7 thermodynamic AHU/VAV/hot-water controller; corrected outside-temperature decoding |
| `_time_key / _advance_clock` | Decode daily phase and day class, guard sequence continuity |
| `_new_day / _finish_day` | Record a complete trajectory, update next-day feedforward profile with finite 15-minute response lag |
| `_correct_zone` | Bound corrections in physical actuator fractions, preserving HVAC on/off decisions |
| `TransferPolicy.get_action` | Run baseline first, apply learned correction only under comparable conditions |
| `reset` | Wipe per-building episodic memory |

## Reproducibility

Run `python Co_Kaggle/g8/submissions/test_version19.py`. The notebook is self-contained and ends with the official `SmartBuildingsInferenceServer(TransferPolicy).run()` entrypoint.

For a fair comparison, run identical **30 scenarios × 8,640 steps** on the official organizer simulator with V19, V15 and V7. Record mean loss and zone comfort tails, energy/carbon, day-wise scores and ablations (ILC off; weather gating off; only cooling; only heating). Synthetic tests establish the interface and learning behavior, **not** a leaderboard win.

In [ ]:
"""Physics feedback foundation for Version 19 iterative learning control.

Uses the fixed, previously documented physics-gain defaults and corrected
outdoor-temperature normalization from Version 15. No optimizer is run.

Runtime: O(number of AHUs + number of zones) per five-minute step.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


# Fixed physical-gain guardrails inherited from V15; V19 does no offline tuning.
PARAMETER_BOUNDS = {
    "active_threshold_k": (0.08, 0.55),
    "cooling_offset_k": (2.0, 7.0),
    "cooling_error_gain": (0.55, 2.2),
    "cool_airflow_gain": (1.0, 3.4),
    "reheat_gain": (0.15, 0.55),
    "fan_pressure_gain_pa": (800.0, 1700.0),
    "water_weather_gain": (0.15, 0.75),
    "water_shortfall_gain": (3.0, 8.0),
}
DEFAULT_PARAMETERS = {"active_threshold_k": 0.18, "cooling_offset_k": 4.0, "cooling_error_gain": 1.3, "cool_airflow_gain": 2.1, "reheat_gain": 0.31, "fan_pressure_gain_pa": 1250.0, "water_weather_gain": 0.35, "water_shortfall_gain": 5.0}


class PhysicsBaselinePolicy(smart_buildings_inference_server.Policy):
    """Weather-compensated, thermodynamics-aware air and hot-water dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15
    PARAMETERS = DEFAULT_PARAMETERS

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.p = {name: float(self.PARAMETERS[name]) for name in PARAMETER_BOUNDS}
        for name, (low, high) in PARAMETER_BOUNDS.items():
            if not np.isfinite(self.p[name]) or not low <= self.p[name] <= high:
                raise ValueError("Invalid tuned parameter: " + name)
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        outdoor_stats = meta.get("observation_normalization_metadata", {}).get(
            "outside_air_temperature_sensor", {})
        self.outdoor_mean = float(outdoor_stats.get("sample_mean", float("nan")))
        self.outdoor_std = float(outdoor_stats.get("sample_std", float("nan")))

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """No episode state: decisions depend only on the current observation."""
        return None

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _outside_temperature(self, obs, index):
        """Denormalize outdoor air; never treat a z-score as Kelvin."""
        raw = self._value(obs, index)
        if not np.isfinite(raw):
            return float("nan")
        if (np.isfinite(self.outdoor_mean) and
                np.isfinite(self.outdoor_std) and self.outdoor_std > 0.0):
            kelvin = raw * self.outdoor_std + self.outdoor_mean
        else:
            kelvin = raw
        return kelvin if 210.0 <= kelvin <= 330.0 else float("nan")

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "low": low,
            "high": high,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["hot"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.p["active_threshold_k"]:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["hot"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - self.p["cooling_offset_k"] - self.p["cooling_error_gain"] * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.p["active_threshold_k"]
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Size VAV flow using actual heating/cooling temperature headroom.

        In contrast to a fixed damper = gain * temperature-error rule,
        requested damper opening falls as cooling headroom grows. This is
        a *proxy* for delivered flow, not a calibrated CFM calculation.
        Returns (mean_damper, peak_damper) for AHU fan coordination.
        """
        sum_damper = 0.0
        peak_damper = 0.0
        for status in statuses:
            zone = status["actions"]
            hot, cold, temperature = (
                status["hot"], status["cold"], status["temp"])
            if hot >= self.p["active_threshold_k"]:
                headroom = max(2.5, temperature - delivered)
                damper = self._clamp(0.12 + self.p["cool_airflow_gain"] * hot / headroom,
                                     self.MIN_VAV_FRACTION, 0.98)
            elif cold >= self.p["active_threshold_k"]:
                # Heating is VAV reheat, never the AHU cooling coil.
                damper = self._clamp(0.22 + 0.21 * cold, 0.18, 0.88)
                if zone["valve"] is not None:
                    valve = self._clamp(0.16 + self.p["reheat_gain"] * cold, 0.0, 1.0)
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append((cold, valve))
            else:
                damper = self.MIN_VAV_FRACTION
            sum_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        # Position fractions are proxies, not calibrated volumetric flow.
        return sum_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        worst_violation_k = max(max(z["hot"], z["cold"]) for z in statuses)
        if worst_violation_k < self.p["active_threshold_k"]:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + self.p["fan_pressure_gain_pa"] * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + self.p["water_weather_gain"] * max(0.0, 294.0 - outside) + self.p["water_shortfall_gain"] * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Read zones -> plan AHU air -> set VAVs -> coordinate shared water.

        Performs one pass per AHU and zone; no search, learned model, or
        state carried between five-minute decisions.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            statuses = [
                self._read_zone(obs, z, fallback) for z in ahu["zones"]
            ]
            statuses = [s for s in statuses if s is not None]
            outdoor = self._outside_temperature(obs, ahu["outdoor"])
            if np.isfinite(outdoor) and 210.0 <= outdoor <= 330.0:
                outdoors.append(outdoor)
            else:
                outdoor = float("nan")
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
class TransferPolicy(PhysicsBaselinePolicy):
    """Day-to-day, time-indexed iterative learning around physics feedback.

    Update only at the end of a sufficiently complete day, using the
    temperature measured 15 minutes after each earlier control command.
    The learned profiles correct VAV cooling dampers or heating reheat
    valves, never replace the within-day physical feedback controller.
    """

    SLOTS_PER_DAY = 288                 # 5-minute samples over 24 hours
    RESPONSE_LAG = 3                    # learn from the temperature 15 min later
    MIN_COMPLETE_SAMPLES = 240
    LEARNING_RATE = 0.035               # actuator fraction / kelvin
    RETENTION = 0.90                    # forget stale profile corrections
    MAX_COOL_CORRECTION = 0.15          # actuator fractions, not action units
    MAX_HEAT_CORRECTION = 0.18
    WEATHER_TOLERANCE_K = 4.0
    TARGET_MARGIN_K = 0.20

    def reset(self):
        """Forget episode-specific learning; do not leak hidden scenarios.

        The physics base invokes reset() after indexing its AHUs and zones.
        This override is therefore also valid during construction.
        """
        self.learning_zones = [
            (ahu, zone) for ahu in self.ahus for zone in ahu["zones"]
        ]
        shape = (len(self.learning_zones), self.SLOTS_PER_DAY)
        self.profiles = {
            day_type: {
                "cool": np.zeros(shape, dtype=np.float32),
                "heat": np.zeros(shape, dtype=np.float32),
                "outdoor": np.full(shape, np.nan, dtype=np.float32),
                "ready": np.zeros(shape, dtype=bool),
            }
            for day_type in ("weekday", "weekend")
        }
        self.day_data = None
        self.last_slot = None

    def _time_key(self, obs):
        """Decode the shared time-of-day and weekday signals, if usable."""
        def angle(sine_name, cosine_name):
            sine = self._value(obs, self.fields.get(sine_name))
            cosine = self._value(obs, self.fields.get(cosine_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            if not 0.5 <= sine * sine + cosine * cosine <= 1.5:
                return None
            return float(np.arctan2(sine, cosine) % (2.0 * np.pi))

        hour = angle("hod_sin_000", "hod_cos_000")
        day_of_week = angle("dow_sin_000", "dow_cos_000")
        if hour is None or day_of_week is None:
            return None
        slot = int(round(hour * self.SLOTS_PER_DAY / (2.0 * np.pi))) % self.SLOTS_PER_DAY
        day = day_of_week * 7.0 / (2.0 * np.pi)
        return slot, ("weekday" if day < 5.0 else "weekend")

    def _new_day(self, day_type):
        shape = (len(self.learning_zones), self.SLOTS_PER_DAY)
        self.day_data = {
            "class": day_type,
            "mode": np.zeros(shape, dtype=np.int8),
            "temperature": np.full(shape, np.nan, dtype=np.float32),
            "low": np.full(shape, np.nan, dtype=np.float32),
            "high": np.full(shape, np.nan, dtype=np.float32),
            "outdoor": np.full(shape, np.nan, dtype=np.float32),
            "applied": np.zeros(shape, dtype=np.float32),
            "visited": np.zeros(self.SLOTS_PER_DAY, dtype=bool),
        }

    def _finish_day(self):
        """Update next day's learned feedforward using an aligned prior day.

        Mode +1 means cooling through the VAV damper; -1 means heating
        through its reheat valve. Positive tracking error demands more
        actuation in either mode. Invalid or incomplete days do not train.
        """
        data = self.day_data
        if data is None or int(data["visited"].sum()) < self.MIN_COMPLETE_SAMPLES:
            return

        profile = self.profiles[data["class"]]
        # Rebuild the next trajectory exclusively from actually EXECUTED
        # feedforward, not the stored proposal (which weather gating or
        # actuator saturation may have prevented us from applying).
        next_cool = np.zeros_like(profile["cool"])
        next_heat = np.zeros_like(profile["heat"])
        next_outdoor = np.full_like(profile["outdoor"], np.nan)
        next_ready = np.zeros_like(profile["ready"])

        # There are deliberately no updates for the final RESPONSE_LAG
        # slots: their response belongs to the next day.
        for zone_index in range(len(self.learning_zones)):
            for slot in range(self.SLOTS_PER_DAY - self.RESPONSE_LAG):
                if not (data["visited"][slot] and
                        data["visited"][slot + self.RESPONSE_LAG]):
                    continue
                mode = int(data["mode"][zone_index, slot])
                if mode == 0:
                    continue
                result_slot = slot + self.RESPONSE_LAG
                temperature = float(data["temperature"][zone_index, result_slot])
                low = float(data["low"][zone_index, result_slot])
                high = float(data["high"][zone_index, result_slot])
                outside = float(data["outdoor"][zone_index, slot])
                if not all(np.isfinite(x) for x in (temperature, low, high, outside)):
                    continue

                if mode == 1:
                    error = temperature - (high - self.TARGET_MARGIN_K)
                    name, limit = "cool", self.MAX_COOL_CORRECTION
                else:
                    error = (low + self.TARGET_MARGIN_K) - temperature
                    name, limit = "heat", self.MAX_HEAT_CORRECTION

                error = float(np.clip(error, -2.5, 2.5))
                applied = float(data["applied"][zone_index, slot])
                new_value = self.RETENTION * applied + self.LEARNING_RATE * error
                target = next_cool if name == "cool" else next_heat
                target[zone_index, slot] = float(np.clip(new_value, -limit, limit))
                next_outdoor[zone_index, slot] = outside
                next_ready[zone_index, slot] = True

        profile["cool"][:] = next_cool
        profile["heat"][:] = next_heat
        profile["outdoor"][:] = next_outdoor
        profile["ready"][:] = next_ready

    def _advance_clock(self, slot, day_type):
        """Only train on sequential, complete-enough days; reject jumps."""
        if self.last_slot is None:
            self._new_day(day_type)
        elif slot < self.last_slot - self.SLOTS_PER_DAY // 2:
            self._finish_day()
            self._new_day(day_type)
        elif slot not in (self.last_slot, self.last_slot + 1):
            # A missing/out-of-order clock invalidates the current trace.
            self._new_day(day_type)
        elif self.day_data["class"] != day_type:
            # An unexpected weekday change is not a verified 24h boundary.
            self._new_day(day_type)
        self.last_slot = slot

    def _correct_zone(self, command, zone, mode, correction):
        """Apply one bounded feedforward offset in physical actuator units."""
        actuator = zone["damper"] if mode == 1 else zone["valve"]
        if actuator is None or not 0 <= actuator < len(command):
            return 0.0
        original = 0.5 * (float(command[actuator]) + 1.0)
        if mode == 1:
            fraction = self._clamp(
                original + correction, self.MIN_VAV_FRACTION, 0.98)
        else:
            fraction = self._clamp(original + correction, 0.0, 1.0)
        command[actuator] = self._fraction_action(fraction)
        return fraction - original

    def get_action(self, obs):
        """Compute safe physics actions, then add gated previous-day ILC."""
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        actions = super().get_action(obs)
        time_key = self._time_key(obs)
        if time_key is None:
            # Missing/invalid time cannot be aligned to a repeated task.
            self.day_data, self.last_slot = None, None
            return actions

        slot, day_type = time_key
        self._advance_clock(slot, day_type)
        data = self.day_data
        profile = self.profiles[day_type]
        fallback = self._schedule_fallback(obs)
        discrete = actions["discrete_action"]
        continuous = actions["continuous_action"]

        for i, (ahu, zone) in enumerate(self.learning_zones):
            status = self._read_zone(obs, zone, fallback)
            if status is None:
                continue
            outside = self._outside_temperature(obs, ahu["outdoor"])
            data["temperature"][i, slot] = status["temp"]
            data["low"][i, slot] = status["low"]
            data["high"][i, slot] = status["high"]
            data["outdoor"][i, slot] = outside

            mode = 0
            threshold = self.p["active_threshold_k"]
            if (status["hot"] >= threshold and zone["damper"] is not None
                    and discrete[ahu["run"]] == 1):
                mode = 1
            elif (status["cold"] >= threshold and zone["valve"] is not None
                    and 0 <= self.water_run < len(discrete)
                    and discrete[self.water_run] == 1):
                mode = -1
            data["mode"][i, slot] = mode

            if mode == 0 or not profile["ready"][i, slot]:
                continue
            reference_outdoor = float(profile["outdoor"][i, slot])
            if not (np.isfinite(outside) and np.isfinite(reference_outdoor)
                    and abs(outside - reference_outdoor) <= self.WEATHER_TOLERANCE_K):
                continue
            name = "cool" if mode == 1 else "heat"
            data["applied"][i, slot] = self._correct_zone(
                continuous, zone, mode, float(profile[name][i, slot]))

        data["visited"][slot] = True
        return actions


In [ ]:
# Official Kaggle gateway; no file-based prediction submission.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
